# 07 - LoRA Adapter Merging

This notebook creates merged LoRA adapters using different model merging strategies:

- Task Arithmetic;
- TIES;
- Model Soup;
- DARE + Task Arithmetic;
- DARE + TIES;
- DARE + Model Soup.

Output adapters are stored as: `artifacts/merged_adapters/<scenario>/<method>/seed_<seed>/adapter_model.safetensors`.

In [ ]:
import exp_common as E
from pathlib import Path

RUN_SEEDS = [42, 123, 777]
ALPHA = 0.8
TOP_K = 0.2
DROP_RATE = 0.5
METHODS = ['task_arithmetic', 'ties', 'model_soup', 'dare+task_arithmetic', 'dare+ties', 'dare+model_soup']

In [ ]:
def merge_scenario(seed, scenario, adapter_dirs):
    for method in METHODS:
        safe = method.replace('+', '_plus_')
        out = E.MERGED_DIR / scenario / safe / f'seed_{seed}'
        print('Merging', scenario, method, 'seed', seed)
        E.merge_adapters(adapter_dirs, out, method=method, seed=seed, alpha=ALPHA, top_k=TOP_K, drop_rate=DROP_RATE)

for seed in RUN_SEEDS:
    logs = E.ADAPTERS_DIR/'logs'/f'seed_{seed}'/'adapter'
    qa = E.ADAPTERS_DIR/'qa'/f'seed_{seed}'/'adapter'
    cpt = E.ADAPTERS_DIR/'cpt'/f'seed_{seed}'/'adapter'
    for p in [logs, qa]:
        if not (p/'adapter_model.safetensors').exists():
            raise FileNotFoundError(p)

    # FT merging: TV_logs + TV_QA in adapter space.
    merge_scenario(seed, 'ft_logs_plus_qa', [logs, qa])

    # CPT + FT merging: CPT adapter + Logs adapter + QA adapter.
    # This is an adapter-space approximation for simultaneous merging.
    if (cpt/'adapter_model.safetensors').exists():
        merge_scenario(seed, 'cpt_plus_logs_plus_qa', [cpt, logs, qa])
    else:
        print('CPT missing, skip CPT+FT for seed', seed)
print('Done')

In [ ]:
# Sequential merging variants: output of previous merge becomes input of next merge.
# Sequential merging variants are implemented as adapter-space averaging / Task Arithmetic operations.
SEQUENTIAL = {
    'seq_logs_then_qa_then_cpt': ['logs', 'qa', 'cpt'],
    'seq_cpt_then_logs_then_qa': ['cpt', 'logs', 'qa'],
    'seq_cpt_then_qa_then_logs': ['cpt', 'qa', 'logs'],
}
BASE_METHOD = 'task_arithmetic'
for seed in RUN_SEEDS:
    mapping = {
        'logs': E.ADAPTERS_DIR/'logs'/f'seed_{seed}'/'adapter',
        'qa': E.ADAPTERS_DIR/'qa'/f'seed_{seed}'/'adapter',
        'cpt': E.ADAPTERS_DIR/'cpt'/f'seed_{seed}'/'adapter',
    }
    for scenario, order in SEQUENTIAL.items():
        if not all((mapping[x]/'adapter_model.safetensors').exists() for x in order):
            print('Skip sequential missing adapter', scenario, seed)
            continue
        current = mapping[order[0]]
        for step, nxt in enumerate(order[1:], start=1):
            out = E.MERGED_DIR / scenario / BASE_METHOD / f'seed_{seed}' / f'step_{step}_{order[step-1]}_plus_{nxt}'
            if not (out/'adapter_model.safetensors').exists():
                E.merge_adapters([current, mapping[nxt]], out, method=BASE_METHOD, seed=seed, alpha=ALPHA)
            current = out
        final = E.MERGED_DIR / scenario / BASE_METHOD / f'seed_{seed}'
        final.mkdir(parents=True, exist_ok=True)
        # copy final adapter files one level up for eval convenience
        import shutil
        for fp in current.iterdir():
            if fp.is_file():
                shutil.copy2(fp, final/fp.name)
        print('Sequential saved:', final)